# 02. Design and evaluate a sales agent

Build an agent that answers questions about the top sales region for a single month. Start with the expected behavior, then check the tool, the model's tool call, and the answer.

Use the [module setup](README.md#setup) and leave Ollama running with `qwen3.5:4b`. The report is included in the notebook. Run the cells from the top. This notebook calls Ollama directly, so the FastAPI server can stay off.

## 1. Work out the answer first

Priya asks, “Which region had the highest sales in March?” Add the March rows by region before you run the code.

| Month | Region | Sales |
|---|---|---:|
| March | North | 700 |
| March | South | 900 |
| March | East | 600 |
| March | North | 500 |
| February | South | 1,800 |

North leads with $1,200. The February row belongs to a different month.

| Situation | Expected behavior |
|---|---|
| March has one winner | Use the March totals and report North and $1,200. |
| The question omits the month | Ask which month to check, with zero tool calls. |
| The report has no April rows | Say April data is unavailable. |
| Two regions tie | Name both regions and give their shared total. |

Use the model to interpret the question and write the reply. Use Python to add the sales.

## 2. Run the calculation on its own

`DictReader` reads the CSV into a list of dictionaries. Print the first row and look at the type of the sales value.

In [ ]:
from csv import DictReader
from io import StringIO

sales_csv = """month,region,sales
March,North,700
March,South,900
March,East,600
March,North,500
February,South,1800
"""

sales_rows = list(DictReader(StringIO(sales_csv)))
print(sales_rows[0])

CSV values arrive as text. Convert each sales value to an integer before adding it. Run the tool for March and April to see what it returns when the report has no matching rows.

In [ ]:
from langchain_core.tools import tool


def make_sales_tool(rows):
    @tool
    def sales_by_region(month: str) -> dict[str, int]:
        """Return sales totals by region for the given month. Use a full month name."""
        totals = {}
        requested_month = month.strip().lower()

        for row in rows:
            if row["month"].lower() == requested_month:
                region = row["region"]
                sales = int(row["sales"])
                # A region starts at zero the first time we see it.
                totals[region] = totals.get(region, 0) + sales

        return totals

    return sales_by_region


sales_by_region = make_sales_tool(sales_rows)
print("March:", sales_by_region.invoke({"month": "March"}))
print("April:", sales_by_region.invoke({"month": "April"}))

March returns three totals. April returns an empty dictionary. Check the calculation here if any of the totals look wrong.

`totals.get(region, 0)` starts a region's running total at zero. `make_sales_tool` also lets you provide a separate set of rows for the tie test.

### Connect the tool to Qwen

`create_agent` gives the model the tool definition. When the model requests the tool, LangChain runs it and sends the result back for the final reply. Read the system prompt and find the line that corresponds to each rule from step 1.

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama


model = ChatOllama(model="qwen3.5:4b", temperature=0, reasoning=False)

missing_month_rule = "If no month is given, ask 'Which month should I check?' and stop."

system_prompt = f"""You answer questions about the top sales region in a single month.
{missing_month_rule}
If the user gives a month, call sales_by_region once with the full month name.
Use the returned totals to find the highest sales. Use only the tool result.
If the tool returns an empty dictionary, say 'No sales data is available for [month].'
Report the winning region and amount in one sentence. If there is a tie for highest
sales, say they tied and name all the winning regions with their shared amount.
"""


def make_agent(sales_tool, prompt=system_prompt):
    return create_agent(
        model=model,
        tools=[sales_tool],
        system_prompt=prompt,
    )


sales_agent = make_agent(sales_by_region)

This notebook uses `invoke()`, which waits for the run and returns its result. The async route in the API uses `await agent.ainvoke()`. Each question here starts with a fresh message list. In the returned run, a model message can contain a tool request and a `ToolMessage` carries the tool's result. The helper below returns the calls, results, and final answer separately.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage


def ask_sales_agent(agent, question):
    run = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 10},
    )
    calls = []
    tool_results = []

    for message in run["messages"]:
        if isinstance(message, AIMessage):
            calls.extend(message.tool_calls)
        if isinstance(message, ToolMessage):
            tool_results.append(message.content)

    return {
        "calls": calls,
        "tool_results": tool_results,
        "answer": run["messages"][-1].text,
    }


question = "Which region had the highest sales in March?"
example = ask_sales_agent(sales_agent, question)

for call in example["calls"]:
    print("Tool call", call["name"], call["args"])

print("Tool returned", example["tool_results"])
print("Answer", example["answer"])

Look for `sales_by_region` with March, followed by the three totals and a reply naming North. If the reply names the correct region but the tool was called with February, the run still misses the requirement.

**Try it.** Ask the same March question in your own words and inspect the tool arguments.

## 3. Turn those expectations into evals

Start with two checks on the March example. Did it use the right tool and month? Does the reply include North and $1,200?

In [ ]:
called_march = (
    len(example["calls"]) == 1
    and example["calls"][0]["name"] == "sales_by_region"
    and example["calls"][0]["args"].get("month", "").lower() == "march"
)

answer_text = example["answer"].lower().replace(",", "")
print("Right tool and month?", called_march)
print("Reply mentions North and 1200?", "north" in answer_text and "1200" in answer_text)

Both values should be `True`. Removing the comma allows both `1,200` and `1200`.

Next, add a $300 South sale to a separate copy of the report. North and South now both have $1,200. Only the tie case uses this report.

In [ ]:
tie_rows = sales_rows + [{"month": "March", "region": "South", "sales": "300"}]
tie_agent = make_agent(make_sales_tool(tie_rows))

cases = [
    {
        "name": "March has one winner",
        "question": "Which region led in March?",
        "month": "March",
        "kind": "winner",
        "winners": {"north"},
        "amount": 1200,
    },
    {
        "name": "Month is missing",
        "question": "Which region led?",
        "month": None,
        "kind": "clarify",
    },
    {
        "name": "April is missing from the report",
        "question": "Which region led in April?",
        "month": "April",
        "kind": "no_data",
    },
    {
        "name": "March has a tie",
        "question": "Which region led in March?",
        "month": "March",
        "kind": "tie",
        "winners": {"north", "south"},
        "amount": 1200,
    },
]

Each case has an expected tool month. `None` means the agent should ask for a month with no tool calls. The winner and tie cases also contain the expected regions and amount.

In [ ]:
def check_tool_use(run, expected_month):
    # A tool call is a request from the model.
    # A tool result is the data returned after the tool runs.
    calls = run["calls"]
    results = run["tool_results"]

    if expected_month is None:
        # The user left out the month. The agent should ask which month
        # to check, so we expect zero tool calls and zero tool results.
        return len(calls) == 0 and len(results) == 0

    # When the user gives a month, we expect exactly one call and one result.
    if len(calls) != 1 or len(results) != 1:
        return False

    # We know there is exactly one call, so take the first item.
    call = calls[0]

    # Read the month the model sent to the tool. Use "" if the key is missing.
    # Remove extra spaces and use lowercase so " March " matches "march".
    month_used = str(call["args"].get("month", "")).strip().lower()

    # The check passes when both the tool name and the month are correct.
    return call["name"] == "sales_by_region" and month_used == expected_month.lower()

The tool check counts calls and results, then compares the tool name and month. When `expected_month` is `None`, both lists should be empty. The next function grades the answer text.

In [ ]:
import re


def check_answer(run, case):
    # Use lowercase so "North" and "north" match.
    # Remove commas so "1,200" and "1200" are treated the same.
    answer = run["answer"].lower().replace(",", "")

    # [a-z]+ finds one or more letters in a row.
    # A set keeps each word only once.
    words = set(re.findall(r"[a-z]+", answer))

    # \d+ finds one or more digits, and \b marks a word boundary.
    # Convert each match from text to an integer for comparison.
    numbers = {int(number) for number in re.findall(r"\b\d+\b", answer)}

    # & gives us the words that appear in both sets.
    # This tells us which of our three regions the answer mentions.
    named_regions = words & {"north", "south", "east"}

    if case["kind"] == "clarify":
        # We expect a question about the month with no numbers or region names.
        return (
            "month" in words
            and "?" in answer
            and not numbers
            and not named_regions
        )

    if case["kind"] == "no_data":
        # We expect the requested month and a message saying data is missing.
        # The answer should contain no sales figures or region names.
        month = case["month"].lower()
        says_no_data = "no sales data" in answer or "no data" in answer
        return (
            month in words
            and says_no_data
            and not numbers
            and not named_regions
        )

    # Check that the reply names exactly the expected winning regions.
    winners_match = named_regions == case["winners"]

    # Check that the expected sales amount is the only distinct number.
    # For example, {1200} passes when the expected amount is 1200.
    amount_matches = numbers == {case["amount"]}

    if case["kind"] == "tie":
        # A tie answer must also explicitly say "tie" or "tied".
        mentions_tie = bool(words & {"tie", "tied"})
        return winners_match and amount_matches and mentions_tie

    # For a single winner, both the region and the amount must match.
    return winners_match and amount_matches

`re.findall` pulls out words and numbers. Set intersection finds which region names were mentioned. This is a small text check. For example, “North did not win with 1,200” can still contain both expected values. If you need a stronger check, validate structured winner and amount fields.

Run the suite and inspect the calls and responses for any case marked `FAIL`.

In [ ]:
def run_evals(agent, agent_with_tie, ask=ask_sales_agent):
    passed = 0

    for case in cases:
        active_agent = agent_with_tie if case["kind"] == "tie" else agent
        run = ask(active_agent, case["question"])

        tool_ok = check_tool_use(run, case["month"])
        answer_ok = check_answer(run, case)
        if tool_ok and answer_ok:
            passed += 1

        print(case["name"])
        print("  Tool check", "PASS" if tool_ok else "FAIL")
        print("  Answer check", "PASS" if answer_ok else "FAIL")
        print("  Calls", [call["args"] for call in run["calls"]])
        print("  Tool results", run["tool_results"])
        print("  Answer", run["answer"])

    print()
    print(f"{passed}/{len(cases)} cases passed.")


run_evals(sales_agent, tie_agent)

A model may guess a month, look it up, then ask for clarification. The wording would pass the answer check, while the extra lookup would fail the tool check. Your baseline score can vary.

The next cell changes the prompt to make March the default. Keep the test cases as they are and predict what will happen to the missing-month case.

In [ ]:
prompt_with_default = system_prompt.replace(
    missing_month_rule,
    "If no month is given, assume March and call sales_by_region with March.",
)

changed_agent = make_agent(sales_by_region, prompt_with_default)
changed_tie_agent = make_agent(make_sales_tool(tie_rows), prompt_with_default)

run_evals(changed_agent, changed_tie_agent)

The changed prompt can produce a March answer even though the requirement says to ask. Change the implementation to satisfy the requirement. The next cell checks for a month before invoking the model.

In [ ]:
from calendar import month_name


def ask_with_month_check(agent, question):
    words = set(re.findall(r"[a-z]+", question.lower()))
    months = {name.lower() for name in list(month_name)[1:]}

    if words.isdisjoint(months):
        # Return the clarification before calling the model or the tool.
        return {
            "calls": [],
            "tool_results": [],
            "answer": "Which month should I check?",
        }

    return ask_sales_agent(agent, question)


run_evals(sales_agent, tie_agent, ask_with_month_check)

The missing-month case should now show an empty call list and a clarification. Rerun the other cases to check that the fix preserved the expected behavior.

The month check looks for whole words. It treats `may` in “May I ask who led?” as a month and asks for a month name for relative phrases like “last month.” Add one of these to your case list if you want to extend the design.

## 4. Extend the requirements

Priya asks, “Compare March and April.” Write an eval before changing the agent. Which months should the tool receive? What should the reply say about April?

<details><summary>Check your answer</summary>

Check that the run looks up both months and explains that the report has no April data. The current agent handles one month. [Lesson 04](04_sales_agent_api.md) adds a report agent with a comparison tool.

</details>

Next, open [Lesson 03](03_agent_reliability.ipynb) to test a tool failure and an approval gate.